# 01 · Raw ROI time-series extraction (Schaefer-100) — UCLA CNP / OpenNeuro ds000030

**Purpose.** Heavy step of the revised analysis. For every participant with schizophrenia (SCHZ) or
healthy control (CONTROL) status and a resting-state scan, this notebook:

1. reads the fMRIPrep-preprocessed resting-state BOLD image in `MNI152NLin2009cAsym` space and its brain
   mask (public fMRIPrep 0.4.4 derivatives of ds000030, release R1.0.5; Gorgolewski et al., 2017),
2. extracts the **raw** (uncleaned) mean BOLD signal of each of the 100 Schaefer-2018 parcels
   (7-network version), plus a cerebrospinal-fluid (CSF) signal from the subject's CSF probability map,
3. copies the fMRIPrep confounds file and the BIDS sidecar, and
4. writes a small package (`roi_package_v2.zip`, ~20–40 MB) with everything needed for the
   light-weight analyses (denoising strategies, connectivity, machine learning, statistics), which are run
   with `scripts/02_run_all_analyses.py`.

No denoising is applied here, so that all confound-regression strategies can be compared on identical
signals. A validation cell checks that the original time series of the first submission are reproduced
exactly when the original cleaning settings are re-applied to these raw signals.

---
**Instrucciones (español).** Abra este cuaderno en Google Colab con la misma cuenta de Google donde está la
carpeta `cnp_ds000030_subset`. Use *Entorno de ejecución → Ejecutar todo*. Autorice el acceso a Google Drive
cuando se le pida. Tarda aproximadamente 45–90 minutos. Si Colab se desconecta, vuelva a ejecutar todo: el
cuaderno continúa donde quedó. Al final, descargue `cnp_ds000030_subset/roi_package_v2.zip`.
Nada de lo existente en la carpeta se modifica: todo se escribe en `_revision_v2/`.

In [ ]:
# 1) Mount Google Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, glob, shutil, platform, datetime

BASE = "/content/drive/MyDrive/cnp_ds000030_subset"    # project folder in Google Drive
DERIV_DRIVE = f"{BASE}/derivates/fmriprep"              # fMRIPrep derivatives already stored in Drive
ORIG_ROI_DIR = f"{BASE}/_roi"                           # time series of the original submission (validation only)
OUT = f"{BASE}/_revision_v2"                            # NEW output folder (nothing existing is modified)
TMP = "/content/tmp_deriv"                              # temporary local downloads, deleted after each subject

for d in [OUT, TMP] + [f"{OUT}/{s}" for s in ("roi_raw", "csf", "confounds", "sidecars", "phenotype", "logs")]:
    os.makedirs(d, exist_ok=True)
print("Project folder found:", os.path.isdir(BASE))
print("Drive derivatives found:", os.path.isdir(DERIV_DRIVE))

In [ ]:
# 2) Software (versions are logged for the Methods section)
!pip -q install nilearn nibabel
import numpy as np, pandas as pd, nibabel as nib, scipy, sklearn, nilearn, requests
from scipy import ndimage
from nilearn import datasets, image, plotting, signal

versions = {
    "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
    "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "nibabel": nib.__version__,
    "nilearn": nilearn.__version__, "run_date_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
}
with open(f"{OUT}/logs/versions_phaseA.json", "w") as f:
    json.dump(versions, f, indent=2)
versions

In [ ]:
# 3) Public metadata from OpenNeuro (small files: participants, phenotype, sidecars)
RAW_URL = "https://s3.amazonaws.com/openneuro.org/ds000030"   # ds000030 v1.0.0 (raw BIDS + phenotype)
DERIV_URL = "https://s3.amazonaws.com/openneuro/ds000030/ds000030_R1.0.5/uncompressed/derivatives/fmriprep"

def download(url, dest, retries=4, timeout=300):
    # Download url to dest. Returns False if the file does not exist (HTTP 404).
    for k in range(retries):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                if r.status_code == 404:
                    return False
                r.raise_for_status()
                part = dest + ".part"
                with open(part, "wb") as fh:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        fh.write(chunk)
                os.replace(part, dest)
                return True
        except Exception as e:
            print(f"   retry {k + 1}/{retries} ({e})")
            time.sleep(5 * (k + 1))
    return False

assert download(f"{RAW_URL}/participants.tsv", f"{OUT}/participants.tsv"), "participants.tsv not downloaded"
download(f"{RAW_URL}/task-rest_bold.json", f"{OUT}/sidecars/task-rest_bold.json")
PHENO = ["demographics", "medication", "bprs", "sans", "saps", "scid", "wais", "wms", "cvlt", "hamilton",
         "ymrs", "handedness", "language", "health", "colortrails", "dkefs"]
for ph in PHENO:
    for ext in ("tsv", "json"):
        download(f"{RAW_URL}/phenotype/{ph}.{ext}", f"{OUT}/phenotype/{ph}.{ext}")

orig_cohort = f"{BASE}/participants_eszvscontrol.tsv"   # manually selected cohort of the first submission
if os.path.isfile(orig_cohort):
    shutil.copy(orig_cohort, f"{OUT}/participants_original_manual_cohort.tsv")

parts = pd.read_csv(f"{OUT}/participants.tsv", sep="\t")
elig = parts[parts["diagnosis"].isin(["SCHZ", "CONTROL"]) & (parts["rest"] == 1)].reset_index(drop=True)
print("Participants with a resting-state scan:")
print(elig["diagnosis"].value_counts())

In [ ]:
# 4) Schaefer-2018 atlas (100 parcels, 7 networks, 2 mm) + parcel labels and centroids
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=2)
atlas_img = image.load_img(atlas.maps)
labels = [l.decode() if isinstance(l, bytes) else str(l) for l in atlas.labels]
labels = [l for l in labels if l.strip().lower() != "background"]   # recent nilearn versions prepend 'Background'
assert len(labels) == 100, f"expected 100 parcel labels, got {len(labels)}"
values = set(np.unique(np.asarray(atlas_img.dataobj)).astype(int).tolist())
assert values == set(range(101)), "unexpected label values in the atlas image"

coords = plotting.find_parcellation_cut_coords(labels_img=atlas_img)   # one row per label value 1..100
assert coords.shape == (100, 3)

def _split(lab):
    parts_ = lab.replace("7Networks_", "").split("_")
    return parts_[0], parts_[1]

atlas_df = pd.DataFrame({
    "roi_index": np.arange(100), "label_value": np.arange(1, 101), "label": labels,
    "hemisphere": [_split(l)[0] for l in labels], "network": [_split(l)[1] for l in labels],
    "x_mni": coords[:, 0], "y_mni": coords[:, 1], "z_mni": coords[:, 2],
})
atlas_df.to_csv(f"{OUT}/atlas_schaefer2018_100p_7n.csv", index=False)
nib.save(atlas_img, f"{OUT}/atlas_schaefer2018_100p_7n_2mm.nii.gz")
atlas_df.head(12)

In [ ]:
# 5) Extraction helpers
def _resample(img, target_affine, target_shape, interpolation):
    try:
        return image.resample_img(img, target_affine=target_affine, target_shape=target_shape,
                                  interpolation=interpolation, force_resample=True, copy_header=True)
    except TypeError:   # older nilearn versions
        return image.resample_img(img, target_affine=target_affine, target_shape=target_shape,
                                  interpolation=interpolation)

_atlas_cache = {}
def atlas_on_grid(affine, shape3d):
    # Atlas labels resampled (nearest neighbour) to the BOLD grid, as NiftiLabelsMasker(resampling_target="data") does.
    key = (tuple(shape3d), np.round(affine, 4).tobytes())
    if key not in _atlas_cache:
        res = _resample(atlas_img, affine, shape3d, "nearest")
        _atlas_cache[key] = np.asarray(res.dataobj).astype(np.int32)
    return _atlas_cache[key]

def extract_subject(bold_path, mask_path, csf_prob_path=None):
    bold = nib.load(bold_path)
    shape3d, affine = bold.shape[:3], bold.affine
    lab = atlas_on_grid(affine, shape3d)
    brain = np.asarray(nib.load(mask_path).dataobj) > 0
    if brain.shape != lab.shape:
        brain = np.asarray(_resample(nib.load(mask_path), affine, shape3d, "nearest").dataobj) > 0
    lab = np.where(brain, lab, 0)

    data = bold.get_fdata(dtype=np.float32)
    n_vols = data.shape[-1]
    flat = data.reshape(-1, n_vols)
    lab_flat = lab.reshape(-1)

    # Raw mean signal per parcel (identical to NiftiLabelsMasker strategy="mean" without cleaning)
    ts = np.full((n_vols, 100), np.nan, dtype=np.float32)
    nvox = np.zeros(100, dtype=int)
    for k in range(1, 101):
        idx = np.flatnonzero(lab_flat == k)
        nvox[k - 1] = idx.size
        if idx.size:
            ts[:, k - 1] = flat[idx].mean(axis=0, dtype=np.float64)

    # CSF signal: CSF probability >= 0.95, eroded once at native resolution, resampled to the BOLD grid
    csf_ts, n_csf, csf_rule = None, 0, "not available"
    if csf_prob_path is not None and os.path.isfile(csf_prob_path):
        csf_img = nib.load(csf_prob_path)
        prob = np.asarray(csf_img.dataobj)
        for thr, n_ero in ((0.95, 1), (0.95, 0), (0.90, 0)):
            binm = prob >= thr
            if n_ero:
                binm = ndimage.binary_erosion(binm, iterations=n_ero)
            on_bold = _resample(nib.Nifti1Image(binm.astype(np.float32), csf_img.affine), affine, shape3d, "linear")
            csf_mask = (np.asarray(on_bold.dataobj) >= 0.5) & brain
            if csf_mask.sum() >= 10:
                n_csf, csf_rule = int(csf_mask.sum()), f"prob>={thr}, erosion={n_ero}"
                csf_ts = flat[np.flatnonzero(csf_mask.reshape(-1))].mean(axis=0, dtype=np.float64)
                break

    zooms = bold.header.get_zooms()
    meta = {"n_vols": int(n_vols), "shape": "x".join(map(str, shape3d)),
            "voxel_mm": "x".join(f"{z:.2f}" for z in zooms[:3]),
            "tr_header": float(zooms[3]) if len(zooms) > 3 else None, "csf_rule": csf_rule}
    return ts, nvox, csf_ts, n_csf, meta

print("Helpers ready.")

In [ ]:
# 6) Main loop: one subject at a time (resumable). Heavy files are never kept.
LOG = f"{OUT}/logs/extraction_log.jsonl"
done = set()
if os.path.isfile(LOG):
    with open(LOG) as fh:
        for line in fh:
            rec = json.loads(line)
            if rec.get("status") == "ok":
                done.add(rec["participant_id"])
print(f"Already extracted: {len(done)} of {len(elig)}")

t_start = time.time()
for n, row in elig.iterrows():
    sid = row["participant_id"]
    if sid in done:
        continue
    t1 = time.time()
    rec = {"participant_id": sid, "diagnosis": row["diagnosis"]}
    sub_tmp = f"{TMP}/{sid}"
    os.makedirs(sub_tmp, exist_ok=True)
    try:
        names = {"bold": f"{sid}_task-rest_bold_space-MNI152NLin2009cAsym_preproc.nii.gz",
                 "mask": f"{sid}_task-rest_bold_space-MNI152NLin2009cAsym_brainmask.nii.gz",
                 "conf": f"{sid}_task-rest_bold_confounds.tsv"}
        drive_func = f"{DERIV_DRIVE}/{sid}/func"
        if all(os.path.isfile(f"{drive_func}/{v}") for v in names.values()):
            paths = {k: f"{drive_func}/{v}" for k, v in names.items()}
            rec["source"] = "google_drive"
        else:
            paths = {k: f"{sub_tmp}/{v}" for k, v in names.items()}
            rec["source"] = "openneuro_R1.0.5"
            for k, v in names.items():
                if not download(f"{DERIV_URL}/{sid}/func/{v}", paths[k]):
                    raise FileNotFoundError(f"{v} is not available in the public fMRIPrep derivatives")

        csf_name = f"{sid}_T1w_space-MNI152NLin2009cAsym_class-CSF_probtissue.nii.gz"
        p_csf = f"{sub_tmp}/{csf_name}"
        has_csf = download(f"{DERIV_URL}/{sid}/anat/{csf_name}", p_csf)
        download(f"{RAW_URL}/{sid}/func/{sid}_task-rest_bold.json", f"{OUT}/sidecars/{sid}_task-rest_bold.json")

        ts, nvox, csf_ts, n_csf, meta = extract_subject(paths["bold"], paths["mask"], p_csf if has_csf else None)
        np.save(f"{OUT}/roi_raw/{sid}_schaefer100_raw.npy", ts)
        if csf_ts is not None:
            np.save(f"{OUT}/csf/{sid}_csf_raw.npy", csf_ts.astype(np.float32))
        shutil.copy(paths["conf"], f"{OUT}/confounds/{names['conf']}")
        rec.update(meta)
        rec.update({"status": "ok", "min_vox_per_roi": int(nvox.min()), "n_empty_rois": int((nvox == 0).sum()),
                    "n_csf_vox": int(n_csf), "nvox_per_roi": nvox.tolist()})
    except Exception as e:
        rec.update({"status": "error", "error": repr(e)})
    finally:
        shutil.rmtree(sub_tmp, ignore_errors=True)
    rec["seconds"] = round(time.time() - t1, 1)
    with open(LOG, "a") as fh:
        fh.write(json.dumps(rec) + "\n")
    print(f"[{n + 1}/{len(elig)}] {sid} {rec['status']:5s} {rec.get('source', '-'):17s} "
          f"{rec['seconds']:6.1f} s | total {(time.time() - t_start) / 60:5.1f} min")
print("Done.")

In [ ]:
# 7) Extraction summary
with open(LOG) as fh:
    log = [json.loads(line) for line in fh]
man = pd.DataFrame(log).drop_duplicates("participant_id", keep="last")
man.drop(columns=["nvox_per_roi"], errors="ignore").to_csv(f"{OUT}/manifest_extraction.csv", index=False)
print(man.groupby(["diagnosis", "status"]).size(), "\n")
ok = man[man["status"] == "ok"]
print("Sources:", ok["source"].value_counts().to_dict())
print("Volumes per run:", ok["n_vols"].value_counts().to_dict())
print("Grid / voxel size:", ok["shape"].unique(), ok["voxel_mm"].unique())
print("Subjects with an empty parcel:", int((ok["n_empty_rois"] > 0).sum()))
print("CSF signal available:", int((ok["n_csf_vox"] > 0).sum()), "of", len(ok))
print("\nNot extracted:")
print(man[man["status"] != "ok"][["participant_id", "diagnosis", "error"]].to_string(index=False))

In [ ]:
# 8) Validation: re-apply the ORIGINAL cleaning to the raw signals and compare with the original time series.
# The original code requested columns named trans_x ... a_comp_cor_XX, which do not exist in these fMRIPrep 0.4.4
# confounds files (X, Y, Z, RotX ..., WhiteMatter, GlobalSignal, aCompCor00 ...). Its fallback then used the first
# 12 numeric columns. Two hypotheses are compared: (a) that fallback, (b) the confounds described in the manuscript.
def conf_fallback12(path):
    cf = pd.read_csv(path, sep="\t")
    cols = cf.select_dtypes(include=["number"]).columns.tolist()[:12]
    return cf[cols].fillna(0.0).to_numpy(), cols

def conf_as_described(path):
    cf = pd.read_csv(path, sep="\t")
    cols = ["X", "Y", "Z", "RotX", "RotY", "RotZ", "WhiteMatter", "GlobalSignal"] + [f"aCompCor0{k}" for k in range(5)]
    return cf[cols].fillna(0.0).to_numpy(), cols

rows = []
for f in sorted(glob.glob(f"{ORIG_ROI_DIR}/*_schaefer100_timeseries.npy")):
    sid = os.path.basename(f).split("_")[0]
    p_raw = f"{OUT}/roi_raw/{sid}_schaefer100_raw.npy"
    p_conf = f"{OUT}/confounds/{sid}_task-rest_bold_confounds.tsv"
    if not (os.path.isfile(p_raw) and os.path.isfile(p_conf)):
        continue
    raw = np.load(p_raw).astype(np.float64)
    orig = np.load(f).astype(np.float64)
    rec = {"participant_id": sid, "same_shape": orig.shape == raw.shape}
    if not rec["same_shape"]:
        rows.append(rec)
        continue
    for name, fn in (("fallback12", conf_fallback12), ("as_described", conf_as_described)):
        C, cols = fn(p_conf)
        rep = signal.clean(raw, detrend=True, standardize="zscore_sample", confounds=C,
                           standardize_confounds=True, filter=False, t_r=2.0)
        rec[f"corr_{name}"] = float(np.corrcoef(rep.ravel(), orig.ravel())[0, 1])
        rec[f"maxabsdiff_{name}"] = float(np.max(np.abs(rep - orig)))
    rows.append(rec)
val = pd.DataFrame(rows)
val.to_csv(f"{OUT}/logs/validation_original_pipeline.csv", index=False)
example_conf = sorted(glob.glob(f"{OUT}/confounds/*_bold_confounds.tsv"))[0]
print("Fallback columns used by the original code:", conf_fallback12(example_conf)[1])
print(f"Subjects compared: {len(val)}")
val.describe().T

In [ ]:
# 9) Package for the light-weight analyses (download this zip and send it / place it in the repository's data/)
pkg = shutil.make_archive(f"{BASE}/roi_package_v2", "zip", root_dir=OUT)
print(pkg, f"{os.path.getsize(pkg) / 1e6:.1f} MB")